In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
customers = pd.read_csv(BASE_PATH + 'customers.csv')

for name, table in [('so_h (sales orders header)', so_h), ('customers', customers)]:
    print(name)
    print('rows:', len(table), '| columns:', len(table.columns))
    print(list(table.columns))
    print(table.head(3))
    print()

Mounted at /content/drive
so_h (sales orders header)
rows: 20000 | columns: 11
['so_id', 'customer_id', 'branch_id', 'order_date', 'delivery_date', 'order_status', 'payment_terms', 'total_order_value', 'total_gst_amount', 'grand_total', 'sales_channel']
       so_id customer_id branch_id  order_date delivery_date order_status  \
0  SO-990591       C0070    AHM001  2022-05-24    2022-05-28    Delivered   
1  SO-431510       C0031    KOL001  2022-05-01    2022-05-08    Delivered   
2  SO-495859       C0013    AHM001  2023-10-01    2023-10-02    Delivered   

  payment_terms  total_order_value  total_gst_amount  grand_total  \
0        Net 30            1345720          374439.6    1720159.6   
1       Advance            2194850          609423.0    2804273.0   
2       Advance            2620800          733824.0    3354624.0   

    sales_channel  
0    Counter Sale  
1  Dealer Network  
2          Online  

customers
rows: 500 | columns: 15
['customer_id', 'customer_type', 'industry_se

In [2]:
# keep only delivered orders and check customers match
sales_h = so_h[so_h['order_status'] == 'Delivered'].copy()
sales_h['order_date'] = pd.to_datetime(sales_h['order_date'])

print('Delivered orders:', len(sales_h))
print('Orders with a blank customer, date or value:',
      sales_h[['customer_id', 'order_date', 'total_order_value']].isnull().any(axis=1).sum())
print('Orders whose customer is not in the customers table:',
      (~sales_h['customer_id'].isin(customers['customer_id'])).sum())
print('First order date:', sales_h['order_date'].min().date())
print('Last order date:', sales_h['order_date'].max().date())
print('Customers in the customers table:', customers['customer_id'].nunique())
print('Customers with at least one delivered order:', sales_h['customer_id'].nunique())
print('Smallest order value:', sales_h['total_order_value'].min())
print('Largest order value:', sales_h['total_order_value'].max())

Delivered orders: 18033
Orders with a blank customer, date or value: 0
Orders whose customer is not in the customers table: 0
First order date: 2019-01-01
Last order date: 2024-12-31
Customers in the customers table: 500
Customers with at least one delivered order: 500
Smallest order value: 310
Largest order value: 8169760


In [3]:
# one row per customer: Recency, Frequency, Monetary (delivered orders only)
snapshot = pd.Timestamp('2025-01-01')

rfm = sales_h.groupby('customer_id').agg(
    last_order=('order_date', 'max'),
    frequency=('so_id', 'count'),
    monetary=('total_order_value', 'sum')
).reset_index()
rfm['recency_days'] = (snapshot - rfm['last_order']).dt.days

print('rfm rows (customers):', len(rfm))
print('Frequency total:', rfm['frequency'].sum(), '| delivered orders:', len(sales_h))
print('Monetary total:', rfm['monetary'].sum(), '| delivered orders total:', sales_h['total_order_value'].sum())
print('Smallest recency (days):', rfm['recency_days'].min())
print('Largest recency (days):', rfm['recency_days'].max())
print('Smallest frequency:', rfm['frequency'].min())
print('Largest frequency:', rfm['frequency'].max())
print('Smallest monetary:', rfm['monetary'].min())
print('Largest monetary:', rfm['monetary'].max())
print(rfm.head(3))

rfm rows (customers): 500
Frequency total: 18033 | delivered orders: 18033
Monetary total: 22958987430 | delivered orders total: 22958987430
Smallest recency (days): 1
Largest recency (days): 359
Smallest frequency: 21
Largest frequency: 56
Smallest monetary: 17721530
Largest monetary: 84966970
  customer_id last_order  frequency  monetary  recency_days
0       C0001 2024-11-15         34  43201680            47
1       C0002 2024-09-07         51  56261100           116
2       C0003 2024-10-30         38  35112260            63


In [4]:
# how spread out are Recency, Frequency and Monetary? (table rfm, 500 customers)
print('rfm rows:', len(rfm))
print(rfm[['recency_days', 'frequency', 'monetary']].describe(percentiles=[0.25, 0.5, 0.75]).round(1))

bands = pd.cut(rfm['recency_days'], bins=[0, 30, 90, 180, 270, 366],
               labels=['1-30 days', '31-90 days', '91-180 days', '181-270 days', '271-366 days'])
print()
print('Customers per recency band:')
print(bands.value_counts().sort_index())

rfm rows: 500
       recency_days  frequency    monetary
count         500.0      500.0       500.0
mean           61.9       36.1  45917974.9
std            55.1        6.0  10298204.4
min             1.0       21.0  17721530.0
25%            21.0       32.0  38446610.0
50%            47.0       36.0  45321980.0
75%            85.0       40.0  52438192.5
max           359.0       56.0  84966970.0

Customers per recency band:
recency_days
1-30 days       167
31-90 days      218
91-180 days      90
181-270 days     20
271-366 days      5
Name: count, dtype: int64


In [5]:
# score 1 to 5 on each metric (table rfm, 500 customers); same value = same score
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

print('rfm rows:', len(rfm))
for score_col, value_col in [('R_score', 'recency_days'), ('F_score', 'frequency'), ('M_score', 'monetary')]:
    print()
    print(score_col, 'groups (based on', value_col + ')')
    print(rfm.groupby(score_col)[value_col].agg(['count', 'min', 'max']).sort_index(ascending=False))

rfm rows: 500

R_score groups (based on recency_days)
         count  min  max
R_score                 
5          101    1   17
4          103   18   37
3           99   38   59
2           98   60   98
1           99   99  359

F_score groups (based on frequency)
         count  min  max
F_score                 
5           92   42   56
4           92   38   41
3          101   35   37
2           96   32   34
1          119   21   31

M_score groups (based on monetary)
         count       min       max
M_score                           
5          100  54324600  84966970
4          100  47587690  54295880
3          100  42934640  47504560
2          100  37042540  42914200
1          100  17721530  37034600


In [6]:
# turn R, F, M scores into named segments (table rfm, 500 customers)
rfm['FM'] = (rfm['F_score'] + rfm['M_score']) / 2

def segment(row):
    r, fm = row['R_score'], row['FM']
    if r >= 4 and fm >= 4:
        return 'Champions'
    if r <= 2 and fm >= 3.5:
        return 'At risk (valuable but quiet)'
    if r <= 2:
        return 'Quiet and low value'
    if r >= 3 and fm >= 3.5:
        return 'Loyal'
    if r >= 4:
        return 'Recent but small'
    return 'Average'

rfm['segment'] = rfm.apply(segment, axis=1)

summary = rfm.groupby('segment').agg(
    customers=('customer_id', 'count'),
    avg_recency_days=('recency_days', 'mean'),
    avg_orders=('frequency', 'mean'),
    avg_spend=('monetary', 'mean'),
    total_spend=('monetary', 'sum')
)
summary['share_of_revenue_%'] = (summary['total_spend'] / rfm['monetary'].sum() * 100).round(2)
print('rfm rows:', len(rfm))
print('Customers in all segments:', summary['customers'].sum())
print(summary.round(1).sort_values('customers', ascending=False))
print()
print(rfm[['customer_id', 'R_score', 'F_score', 'M_score', 'segment']].head(3))

rfm rows: 500
Customers in all segments: 500
                              customers  avg_recency_days  avg_orders  \
segment                                                                 
Quiet and low value                 127             120.1        32.2   
Recent but small                    114              17.6        32.3   
Loyal                                71              38.8        40.2   
At risk (valuable but quiet)         70             103.8        40.8   
Champions                            66              18.0        43.5   
Average                              52              47.9        32.4   

                               avg_spend  total_spend  share_of_revenue_%  
segment                                                                    
Quiet and low value           39616765.0   5031329160                21.9  
Recent but small              39548701.4   4508551960                19.6  
Loyal                         50849015.1   3610280070             

In [7]:
# is the "At risk" silence longer than the customer's own normal rhythm? (table rfm, 500 customers)
first_order = sales_h.groupby('customer_id')['order_date'].min()
rfm['first_order'] = rfm['customer_id'].map(first_order)
rfm['avg_gap_days'] = (rfm['last_order'] - rfm['first_order']).dt.days / (rfm['frequency'] - 1)
rfm['silence_vs_gap'] = rfm['recency_days'] / rfm['avg_gap_days']

print('rfm rows:', len(rfm))
print('Customers with a blank gap or ratio:', rfm[['avg_gap_days', 'silence_vs_gap']].isnull().any(axis=1).sum())

gap_summary = rfm.groupby('segment').agg(
    customers=('customer_id', 'count'),
    median_gap_days=('avg_gap_days', 'median'),
    median_recency_days=('recency_days', 'median'),
    median_ratio=('silence_vs_gap', 'median'),
    customers_ratio_above_2=('silence_vs_gap', lambda s: (s > 2).sum())
)
print(gap_summary.round(2).sort_values('median_ratio', ascending=False))

rfm rows: 500
Customers with a blank gap or ratio: 0
                              customers  median_gap_days  median_recency_days  \
segment                                                                         
At risk (valuable but quiet)         70            51.81                 96.0   
Quiet and low value                 127            64.22                101.0   
Loyal                                71            53.75                 42.0   
Average                              52            66.22                 47.5   
Champions                            66            50.38                 18.5   
Recent but small                    114            67.00                 16.5   

                              median_ratio  customers_ratio_above_2  
segment                                                              
At risk (valuable but quiet)          1.94                       28  
Quiet and low value                   1.61                       39  
Loyal             

In [8]:
# customers quiet for more than twice their own normal gap (table rfm, 500 customers)
late = rfm[rfm['silence_vs_gap'] > 2]
print('rfm rows:', len(rfm), '| customers with ratio above 2:', len(late))

late_summary = late.groupby('segment').agg(
    customers=('customer_id', 'count'),
    total_spend=('monetary', 'sum')
)
late_summary['share_of_all_revenue_%'] = (late_summary['total_spend'] / rfm['monetary'].sum() * 100).round(2)
print(late_summary)
print('Total spend of these customers:', late['monetary'].sum())
print()
print('Top 10 by spend:')
print(late.sort_values('monetary', ascending=False)[
    ['customer_id', 'segment', 'recency_days', 'avg_gap_days', 'silence_vs_gap', 'monetary']
].head(10).round(2))

rfm rows: 500 | customers with ratio above 2: 67
                              customers  total_spend  share_of_all_revenue_%
segment                                                                     
At risk (valuable but quiet)         28   1524468230                    6.64
Quiet and low value                  39   1490920520                    6.49
Total spend of these customers: 3015388750

Top 10 by spend:
    customer_id                       segment  recency_days  avg_gap_days  \
109       C0110  At risk (valuable but quiet)           194         48.15   
253       C0254  At risk (valuable but quiet)            98         40.70   
220       C0221  At risk (valuable but quiet)           118         47.86   
102       C0103  At risk (valuable but quiet)           125         51.69   
153       C0154  At risk (valuable but quiet)           133         46.16   
336       C0337  At risk (valuable but quiet)           129         59.36   
296       C0297  At risk (valuable but quie

In [9]:
# who are the At risk customers? (table rfm joined with customers, 500 rows)
cust = rfm.merge(customers[['customer_id', 'customer_type', 'region', 'branch_id',
                            'payment_terms', 'industry_segment']], on='customer_id', how='left')
cust['at_risk'] = (cust['segment'] == 'At risk (valuable but quiet)')

print('rows:', len(cust), '| customers without details:', cust['customer_type'].isnull().sum())
print('Overall At risk share %:', round(cust['at_risk'].mean() * 100, 1), '(', cust['at_risk'].sum(), 'of', len(cust), ')')

for col in ['customer_type', 'region', 'branch_id', 'payment_terms', 'industry_segment']:
    t = cust.groupby(col).agg(customers=('customer_id', 'count'), at_risk=('at_risk', 'sum'))
    t['at_risk_%'] = (t['at_risk'] / t['customers'] * 100).round(1)
    print()
    print(t.sort_values('at_risk_%', ascending=False))

rows: 500 | customers without details: 0
Overall At risk share %: 14.0 ( 70 of 500 )

               customers  at_risk  at_risk_%
customer_type                               
Dealer                99       19       19.2
Fleet Owner           95       14       14.7
Corporate            104       15       14.4
Government           114       14       12.3
Retail                88        8        9.1

            customers  at_risk  at_risk_%
region                                   
North              82       19       23.2
South             158       22       13.9
West              151       19       12.6
East               87        8        9.2
Central            11        1        9.1
North East         11        1        9.1

           customers  at_risk  at_risk_%
branch_id                               
DEL001            71       20       28.2
AHM001            88       13       14.8
CHN001           102       14       13.7
HYD001            68        8       11.8
KOL001         

In [10]:
# is each group's At risk share different from chance? (table cust, 500 rows, 28 groups)
from scipy.stats import binomtest

overall = cust['at_risk'].mean()
rows = []
for col in ['customer_type', 'region', 'branch_id', 'payment_terms', 'industry_segment']:
    for value, g in cust.groupby(col):
        n, k = len(g), int(g['at_risk'].sum())
        p = binomtest(k, n, overall, alternative='two-sided').pvalue
        rows.append({'attribute': col, 'group': value, 'customers': n, 'at_risk': k,
                     'expected_at_risk': round(n * overall, 1), 'p_value': round(p, 4)})
result = pd.DataFrame(rows).sort_values('p_value')
print('groups tested:', len(result))
print(result.head(8))
print()
print('DEL001 customers by region:')
print(cust[cust['branch_id'] == 'DEL001'].groupby('region').agg(
    customers=('customer_id', 'count'), at_risk=('at_risk', 'sum')))

groups tested: 28
        attribute    group  customers  at_risk  expected_at_risk  p_value
13      branch_id   DEL001         71       20               9.9   0.0017
7          region    North         82       19              11.5   0.0244
16      branch_id   PUN001         75        5              10.5   0.0676
1   customer_type   Dealer         99       19              13.9   0.1462
17  payment_terms  Advance        103        9              14.4   0.1540
20  payment_terms   Net 45        113       21              15.8   0.1737
4   customer_type   Retail         88        8              12.3   0.2194
6          region     East         87        8              12.2   0.2771

DEL001 customers by region:
            customers  at_risk
region                        
East                4        1
North              59       17
North East          1        0
South               5        1
West                2        1


In [11]:
# the Delhi (DEL001) At risk customers (table cust, 500 rows)
d = cust[(cust['branch_id'] == 'DEL001') & (cust['at_risk'])]
print('DEL001 At risk customers:', len(d))
print('Of these, quiet for more than twice their normal gap:', (d['silence_vs_gap'] > 2).sum())
print('Their total spend:', d['monetary'].sum())
print('Share of all revenue %:', round(d['monetary'].sum() / cust['monetary'].sum() * 100, 2))
print('Share of the whole At risk segment spend %:',
      round(d['monetary'].sum() / cust[cust['at_risk']]['monetary'].sum() * 100, 1))
print()
print(d.sort_values('monetary', ascending=False)[
    ['customer_id', 'customer_type', 'region', 'recency_days', 'avg_gap_days', 'silence_vs_gap', 'monetary']
].round(2).to_string(index=False))

DEL001 At risk customers: 20
Of these, quiet for more than twice their normal gap: 9
Their total spend: 1119969650
Share of all revenue %: 4.88
Share of the whole At risk segment spend %: 29.0

customer_id customer_type region  recency_days  avg_gap_days  silence_vs_gap  monetary
      C0288        Dealer  North            75         38.26            1.96  76291160
      C0083        Dealer  North            89         44.57            2.00  70378740
      C0221        Dealer  North           118         47.86            2.47  64336800
      C0246        Retail  North            60         61.97            0.97  60399770
      C0129     Corporate  North            62         60.71            1.02  59941820
      C0154   Fleet Owner  North           133         46.16            2.88  58187170
      C0476   Fleet Owner  North            94         63.21            1.49  57777740
      C0337     Corporate  North           129         59.36            2.17  57585510
      C0342        Deal

In [12]:
# stricter test by branch: customers quiet for more than twice their own normal gap (table cust, 500 rows)
cust['late'] = cust['silence_vs_gap'] > 2
overall_late = cust['late'].mean()
print('Overall share quiet more than twice normal gap %:', round(overall_late * 100, 1),
      '(', cust['late'].sum(), 'of', len(cust), ')')

rows = []
for b, g in cust.groupby('branch_id'):
    n, k = len(g), int(g['late'].sum())
    rows.append({'branch_id': b, 'customers': n, 'quiet_over_2x': k,
                 'share_%': round(k / n * 100, 1), 'expected': round(n * overall_late, 1),
                 'p_value': round(binomtest(k, n, overall_late).pvalue, 4),
                 'median_recency_days': g['recency_days'].median(),
                 'average_FM': round(g['FM'].mean(), 2)})
print(pd.DataFrame(rows).sort_values('p_value').to_string(index=False))

Overall share quiet more than twice normal gap %: 13.4 ( 67 of 500 )
branch_id  customers  quiet_over_2x  share_%  expected  p_value  median_recency_days  average_FM
   PUN001         75              4      5.3      10.0   0.0405                 48.0        2.77
   CHN001        102             20     19.6      13.7   0.0793                 50.0        2.98
   HYD001         68             11     16.2       9.1   0.4766                 47.0        2.92
   AHM001         88             10     11.4      11.8   0.7535                 47.0        2.85
   DEL001         71             10     14.1       9.5   0.8613                 46.0        3.36
   KOL001         96             12     12.5      12.9   1.0000                 44.5        2.83


In [13]:
# separate value mix from quietness, by branch (table cust, 500 rows)
cust['high_value'] = cust['FM'] >= 3.5
cust['quiet_score'] = cust['R_score'] <= 2

rows = []
for b, g in cust.groupby('branch_id'):
    hv = g[g['high_value']]
    rows.append({'branch_id': b, 'customers': len(g),
                 'high_value': len(hv), 'high_value_share_%': round(len(hv) / len(g) * 100, 1),
                 'of_these_quiet': int(hv['quiet_score'].sum()),
                 'quiet_share_of_high_value_%': round(hv['quiet_score'].mean() * 100, 1)})
out = pd.DataFrame(rows)
print(out.to_string(index=False))
print()
print('All branches: high value', cust['high_value'].sum(), 'of', len(cust),
      '| quiet among them %:', round(cust[cust['high_value']]['quiet_score'].mean() * 100, 1))

branch_id  customers  high_value  high_value_share_%  of_these_quiet  quiet_share_of_high_value_%
   AHM001         88          33                37.5              13                         39.4
   CHN001        102          42                41.2              14                         33.3
   DEL001         71          43                60.6              20                         46.5
   HYD001         68          24                35.3               8                         33.3
   KOL001         96          36                37.5              10                         27.8
   PUN001         75          29                38.7               5                         17.2

All branches: high value 207 of 500 | quiet among them %: 33.8


In [14]:
# is each branch's quiet share among high-value customers different from chance? (table cust)
hv_all = cust[cust['high_value']]
overall_q = hv_all['quiet_score'].mean()
print('High-value customers:', len(hv_all), '| overall quiet share %:', round(overall_q * 100, 1))

rows = []
for b, g in hv_all.groupby('branch_id'):
    n, k = len(g), int(g['quiet_score'].sum())
    rows.append({'branch_id': b, 'high_value': n, 'quiet': k,
                 'expected_quiet': round(n * overall_q, 1),
                 'p_value': round(binomtest(k, n, overall_q).pvalue, 4)})
print(pd.DataFrame(rows).sort_values('p_value').to_string(index=False))

High-value customers: 207 | overall quiet share %: 33.8
branch_id  high_value  quiet  expected_quiet  p_value
   PUN001          29      5             9.8   0.0757
   DEL001          43     20            14.5   0.1055
   KOL001          36     10            12.2   0.4869
   AHM001          33     13            11.2   0.5812
   HYD001          24      8             8.1   1.0000
   CHN001          42     14            14.2   1.0000


In [15]:
# is each branch's share of high-value customers different from chance? (table cust, 500 rows)
overall_hv = cust['high_value'].mean()
print('Overall high-value share %:', round(overall_hv * 100, 1), '(', cust['high_value'].sum(), 'of', len(cust), ')')

rows = []
for b, g in cust.groupby('branch_id'):
    n, k = len(g), int(g['high_value'].sum())
    rows.append({'branch_id': b, 'customers': n, 'high_value': k,
                 'share_%': round(k / n * 100, 1), 'expected': round(n * overall_hv, 1),
                 'p_value': round(binomtest(k, n, overall_hv).pvalue, 4)})
print(pd.DataFrame(rows).sort_values('p_value').to_string(index=False))

Overall high-value share %: 41.4 ( 207 of 500 )
branch_id  customers  high_value  share_%  expected  p_value
   DEL001         71          43     60.6      29.4   0.0015
   HYD001         68          24     35.3      28.2   0.3271
   KOL001         96          36     37.5      39.7   0.4695
   AHM001         88          33     37.5      36.4   0.5165
   PUN001         75          29     38.7      31.0   0.7253
   CHN001        102          42     41.2      42.2   1.0000


In [16]:
# do Recency, Frequency and Monetary overlap? (table rfm, 500 customers)
print('rfm rows:', len(rfm))
print(rfm[['recency_days', 'frequency', 'monetary']].corr(method='spearman').round(2))

rfm rows: 500
              recency_days  frequency  monetary
recency_days          1.00      -0.14      -0.1
frequency            -0.14       1.00       0.7
monetary             -0.10       0.70       1.0


In [17]:
# count blank values, empty ids and zero or negative numbers in the tables used
print("Delivered orders table:", len(sales_h), "rows")
for col in ["so_id", "customer_id", "order_date", "total_order_value", "order_status"]:
    print("blank values in", col, ":", sales_h[col].isna().sum())
print("empty-text order ids:", (sales_h["so_id"].astype(str).str.strip() == "").sum())
print("empty-text customer ids:", (sales_h["customer_id"].astype(str).str.strip() == "").sum())
print("order values of zero or less:", (sales_h["total_order_value"] <= 0).sum())

print()
print("Customer scores table:", len(rfm), "rows")
for col in ["customer_id", "recency_days", "frequency", "monetary", "R_score", "F_score", "M_score", "segment"]:
    print("blank values in", col, ":", rfm[col].isna().sum())
for col in ["recency_days", "frequency", "monetary"]:
    print(col, "zero or less:", (rfm[col] <= 0).sum())

print()
print("Customers table:", len(customers), "rows")
print("blank customer ids:", customers["customer_id"].isna().sum())
print("repeated customer ids:", customers["customer_id"].duplicated().sum())
print("customers in the scores table not found in the customers table:", (~rfm["customer_id"].isin(customers["customer_id"])).sum())

Delivered orders table: 18033 rows
blank values in so_id : 0
blank values in customer_id : 0
blank values in order_date : 0
blank values in total_order_value : 0
blank values in order_status : 0
empty-text order ids: 0
empty-text customer ids: 0
order values of zero or less: 0

Customer scores table: 500 rows
blank values in customer_id : 0
blank values in recency_days : 0
blank values in frequency : 0
blank values in monetary : 0
blank values in R_score : 0
blank values in F_score : 0
blank values in M_score : 0
blank values in segment : 0
recency_days zero or less: 0
frequency zero or less: 0
monetary zero or less: 0

Customers table: 500 rows
blank customer ids: 0
repeated customer ids: 0
customers in the scores table not found in the customers table: 0


In [18]:
# look at unusual cases: customers without orders, customers with one order, same-day repeat orders
print("customers in the customers table:", customers["customer_id"].nunique())
print("customers with at least one Delivered order:", sales_h["customer_id"].nunique())
print("customers with no Delivered order:", (~customers["customer_id"].isin(sales_h["customer_id"])).sum())

orders_per_customer = sales_h.groupby("customer_id").size()
print("fewest orders by one customer:", orders_per_customer.min())
print("customers with only one order:", (orders_per_customer == 1).sum())

same_day = sales_h.groupby(["customer_id", "order_date"]).size()
same_day = same_day[same_day > 1]
print("customer and day pairs with more than one order:", len(same_day))
print("extra orders these create:", (same_day - 1).sum())

copies = sales_h.duplicated(subset=["customer_id", "order_date", "total_order_value"], keep=False)
print("orders that match another order on customer, day and value:", copies.sum())

customers in the customers table: 500
customers with at least one Delivered order: 500
customers with no Delivered order: 0
fewest orders by one customer: 21
customers with only one order: 0
customer and day pairs with more than one order: 132
extra orders these create: 133
orders that match another order on customer, day and value: 0


In [19]:
# look for repeated order ids and repeated rows, and the most orders one customer placed on one day
print("rows in Delivered orders table:", len(sales_h))
print("different order ids in Delivered orders table:", sales_h["so_id"].nunique())
print("repeated order ids in Delivered orders table:", sales_h["so_id"].duplicated().sum())
print("repeated order ids in the full orders table:", so_h["so_id"].duplicated().sum())
print("fully identical rows in Delivered orders table:", sales_h.duplicated().sum())
print("most orders by one customer on one day:", sales_h.groupby(["customer_id", "order_date"]).size().max())

rows in Delivered orders table: 18033
different order ids in Delivered orders table: 18033
repeated order ids in Delivered orders table: 0
repeated order ids in the full orders table: 0
fully identical rows in Delivered orders table: 0
most orders by one customer on one day: 3


In [20]:
# add up the customer table and compare it with the Delivered orders table
print("orders in Delivered orders table:", len(sales_h))
print("orders added over all customers:", rfm["frequency"].sum())
print("order value in Delivered orders table:", sales_h["total_order_value"].sum())
print("money added over all customers:", rfm["monetary"].sum())
print("difference between the two money totals:", sales_h["total_order_value"].sum() - rfm["monetary"].sum())
print("revenue total from earlier work:", 22958987430)
print("difference from the earlier revenue total:", rfm["monetary"].sum() - 22958987430)

print()
seg = rfm.groupby("segment").agg(customers=("customer_id", "count"), orders=("frequency", "sum"), money=("monetary", "sum"))
print(seg)
print("customers over the six segments:", seg["customers"].sum())
print("orders over the six segments:", seg["orders"].sum())
print("money over the six segments:", seg["money"].sum())

print()
status_counts = so_h["order_status"].value_counts()
print(status_counts)
print("all statuses added together:", status_counts.sum(), "rows in the full orders table:", len(so_h))

orders in Delivered orders table: 18033
orders added over all customers: 18033
order value in Delivered orders table: 22958987430
money added over all customers: 22958987430
difference between the two money totals: 0
revenue total from earlier work: 22958987430
difference from the earlier revenue total: 0

                              customers  orders       money
segment                                                    
At risk (valuable but quiet)         70    2854  3859948290
Average                              52    1685  2082344070
Champions                            66    2870  3866533880
Loyal                                71    2856  3610280070
Quiet and low value                 127    4089  5031329160
Recent but small                    114    3679  4508551960
customers over the six segments: 500
orders over the six segments: 18033
money over the six segments: 22958987430

order_status
Delivered    18033
Cancelled     1967
Name: count, dtype: int64
all statuses added t

In [21]:
# load the sales order lines and compare each Delivered order's header value with the sum of its lines
so_l = pd.read_csv(BASE_PATH + "sales_orders_lines.csv")
print("rows in sales order lines table:", len(so_l))

lines_d = so_l[so_l["so_id"].isin(sales_h["so_id"])]
print("lines belonging to Delivered orders:", len(lines_d))
print("order lines value added together:", lines_d["line_total"].sum())
print("order header value added together:", sales_h["total_order_value"].sum())
print("difference between the two totals:", lines_d["line_total"].sum() - sales_h["total_order_value"].sum())

per_order = lines_d.groupby("so_id")["line_total"].sum().rename("lines_value")
compare = sales_h.set_index("so_id")[["total_order_value"]].join(per_order)
print("Delivered orders with no lines:", compare["lines_value"].isna().sum())
gap = (compare["total_order_value"] - compare["lines_value"]).abs()
print("orders where the header value differs from the sum of its lines:", (gap > 0.01).sum())
print("largest difference on one order:", gap.max())

rows in sales order lines table: 130402
lines belonging to Delivered orders: 117656
order lines value added together: 22958987430
order header value added together: 22958987430
difference between the two totals: 0
Delivered orders with no lines: 0
orders where the header value differs from the sum of its lines: 0
largest difference on one order: 0


In [22]:
# rebuild recency, order count, money, scores and segments with a plain loop and plain rules, then compare with the customer table
import csv
from datetime import date
import numpy as np

last_order = {}
order_count = {}
money_total = {}
with open(BASE_PATH + "sales_orders_header.csv", newline="") as f:
    for row in csv.DictReader(f):
        if row["order_status"] != "Delivered":
            continue
        c = row["customer_id"]
        d = date.fromisoformat(row["order_date"][:10])
        last_order[c] = max(last_order.get(c, d), d)
        order_count[c] = order_count.get(c, 0) + 1
        money_total[c] = money_total.get(c, 0) + float(row["total_order_value"])

ids = sorted(last_order)
snapshot = date(2025, 1, 1)
rec = np.array([(snapshot - last_order[c]).days for c in ids])
frq = np.array([order_count[c] for c in ids])
mon = np.array([money_total[c] for c in ids])
print("customers rebuilt:", len(ids))

def group_of(values):
    edges = np.quantile(values, [0.2, 0.4, 0.6, 0.8])
    return np.array([1 + sum(v > e for e in edges) for v in values])

r_sc = 6 - group_of(rec)
f_sc = group_of(frq)
m_sc = group_of(mon)

def segment_of(r, f, m):
    fm = (f + m) / 2
    if r >= 4 and fm >= 4:
        return "Champions"
    if r <= 2 and fm >= 3.5:
        return "At risk (valuable but quiet)"
    if r <= 2:
        return "Quiet and low value"
    if r >= 3 and fm >= 3.5:
        return "Loyal"
    if r >= 4:
        return "Recent but small"
    return "Average"

seg = [segment_of(r_sc[i], f_sc[i], m_sc[i]) for i in range(len(ids))]

table = rfm.set_index("customer_id").loc[ids]
print("customers where recency differs:", (table["recency_days"].values != rec).sum())
print("customers where order count differs:", (table["frequency"].values != frq).sum())
print("customers where money differs:", (np.abs(table["monetary"].values - mon) > 0.01).sum())
print("customers where the recency score differs:", (table["R_score"].values != r_sc).sum())
print("customers where the frequency score differs:", (table["F_score"].values != f_sc).sum())
print("customers where the money score differs:", (table["M_score"].values != m_sc).sum())
print("customers where the segment differs:", (table["segment"].values != np.array(seg)).sum())

customers rebuilt: 500
customers where recency differs: 0
customers where order count differs: 0
customers where money differs: 0
customers where the recency score differs: 0
customers where the frequency score differs: 0
customers where the money score differs: 0
customers where the segment differs: 0


In [23]:
# move each segment cut point a little and see how many customers change segment and how stable the At risk group is
def segment_with(r, f, m, champion_fm=4, high_fm=3.5, quiet_r=2):
    fm = (f + m) / 2
    if r >= 4 and fm >= champion_fm:
        return "Champions"
    if r <= quiet_r and fm >= high_fm:
        return "At risk (valuable but quiet)"
    if r <= quiet_r:
        return "Quiet and low value"
    if r >= 3 and fm >= high_fm:
        return "Loyal"
    if r >= 4:
        return "Recent but small"
    return "Average"

r_vals = rfm["R_score"].values
f_vals = rfm["F_score"].values
m_vals = rfm["M_score"].values
now = rfm["segment"].values
at_risk = "At risk (valuable but quiet)"

same_rules = [segment_with(r_vals[i], f_vals[i], m_vals[i]) for i in range(len(rfm))]
print("customers where the normal rules differ from the table:", (np.array(same_rules) != now).sum())
print("customers in the At risk group now:", (now == at_risk).sum())
print()

variants = {
    "Champions need average score 3.5": dict(champion_fm=3.5),
    "Champions need average score 4.5": dict(champion_fm=4.5),
    "valuable starts at 3.0": dict(high_fm=3.0),
    "valuable starts at 4.0": dict(high_fm=4.0),
    "quiet means recency score 1 only": dict(quiet_r=1),
}
for name, change in variants.items():
    new = np.array([segment_with(r_vals[i], f_vals[i], m_vals[i], **change) for i in range(len(rfm))])
    print(name)
    print("  customers who change segment:", (new != now).sum())
    print("  customers in the At risk group:", (new == at_risk).sum())
    print("  of today's At risk customers still At risk:", ((new == at_risk) & (now == at_risk)).sum())

customers where the normal rules differ from the table: 0
customers in the At risk group now: 70

Champions need average score 3.5
  customers who change segment: 24
  customers in the At risk group: 70
  of today's At risk customers still At risk: 70
Champions need average score 4.5
  customers who change segment: 15
  customers in the At risk group: 70
  of today's At risk customers still At risk: 70
valuable starts at 3.0
  customers who change segment: 64
  customers in the At risk group: 92
  of today's At risk customers still At risk: 70
valuable starts at 4.0
  customers who change segment: 62
  customers in the At risk group: 49
  of today's At risk customers still At risk: 49
quiet means recency score 1 only
  customers who change segment: 98
  customers in the At risk group: 33
  of today's At risk customers still At risk: 33


In [24]:
# find the customers who stay At risk under every moved cut point and see how many are also quiet beyond twice their own normal gap
tight = np.array([segment_with(r_vals[i], f_vals[i], m_vals[i], high_fm=4.0, quiet_r=1) for i in range(len(rfm))])
firm = rfm[tight == at_risk].copy()
print("customers who stay At risk under every setting:", len(firm))
print("of these quiet beyond twice their own normal gap:", (firm["silence_vs_gap"] > 2).sum())
print("spend of this group:", firm["monetary"].sum())
print("share of all revenue:", round(firm["monetary"].sum() / rfm["monetary"].sum() * 100, 2), "%")
print()
print(firm.sort_values("monetary", ascending=False)[["customer_id", "recency_days", "avg_gap_days", "silence_vs_gap", "frequency", "monetary"]].to_string(index=False))

customers who stay At risk under every setting: 21
of these quiet beyond twice their own normal gap: 19
spend of this group: 1164484450
share of all revenue: 5.07 %

customer_id  recency_days  avg_gap_days  silence_vs_gap  frequency  monetary
      C0110           194     48.146341        4.029382         42  71230450
      C0221           118     47.860465        2.465500         44  64336800
      C0103           125     51.692308        2.418155         40  59164240
      C0154           133     46.159091        2.881339         45  58187170
      C0084           101     51.921053        1.945261         39  57484480
      C0384           114     53.763158        2.120411         39  56692240
      C0002           116     41.060000        2.825134         51  56261100
      C0489           137     42.434783        3.228484         47  55400350
      C0025           136     54.810811        2.481262         38  55220350
      C0373           121     53.486486        2.262254         

In [25]:
# look at the largest orders and test whether a customer's single biggest order decides their money score or segment
order_values = sales_h["total_order_value"]
print("average order value:", round(order_values.mean()))
print("largest order value:", order_values.max())
print("largest order is", round(order_values.max() / order_values.mean(), 1), "times the average order")
print()
print(sales_h.nlargest(10, "total_order_value")[["so_id", "customer_id", "order_date", "total_order_value"]].to_string(index=False))
print()

by_customer = rfm.set_index("customer_id")
biggest = sales_h.groupby("customer_id")["total_order_value"].max()
share_of_money = biggest / by_customer["monetary"] * 100
print("biggest order as a share of the customer's money: smallest", round(share_of_money.min(), 2), "%, average", round(share_of_money.mean(), 2), "%, largest", round(share_of_money.max(), 2), "%")

money_without = by_customer["monetary"] - biggest
new_m_score = pd.qcut(money_without, 5, labels=False) + 1
new_segment = pd.Series([segment_with(by_customer.loc[c, "R_score"], by_customer.loc[c, "F_score"], new_m_score[c]) for c in by_customer.index], index=by_customer.index)
print("customers whose money score changes without their biggest order:", (new_m_score != by_customer["M_score"]).sum())
print("customers whose segment changes without their biggest order:", (new_segment != by_customer["segment"]).sum())
print("customers in the At risk group without their biggest order:", (new_segment == at_risk).sum())

average order value: 1273165
largest order value: 8169760
largest order is 6.4 times the average order

    so_id customer_id order_date  total_order_value
SO-246196       C0268 2020-02-18            8169760
SO-624611       C0102 2021-02-13            7791450
SO-929145       C0359 2023-04-24            7752220
SO-786016       C0349 2024-10-03            7731900
SO-667806       C0030 2023-07-18            7146240
SO-713731       C0200 2022-11-07            7001250
SO-431481       C0162 2022-05-31            6998530
SO-585493       C0234 2024-10-22            6927700
SO-609293       C0153 2024-02-22            6716250
SO-271147       C0225 2024-07-17            6635650

biggest order as a share of the customer's money: smallest 5.15 %, average 9.43 %, largest 17.38 %
customers whose money score changes without their biggest order: 60
customers whose segment changes without their biggest order: 20
customers in the At risk group without their biggest order: 70


In [26]:
# compare the At risk group with and without each customer's biggest order, and check the firm group
now_set = set(by_customer.index[by_customer["segment"] == at_risk])
new_set = set(new_segment.index[new_segment == at_risk])
print("At risk now:", len(now_set), "and without the biggest orders:", len(new_set))
print("customers in both groups:", len(now_set & new_set))
print("customers who leave the At risk group:", sorted(now_set - new_set))
print("customers who join the At risk group:", sorted(new_set - now_set))
print("firm At risk customers still At risk without their biggest order:", firm["customer_id"].isin(new_set).sum(), "of", len(firm))

At risk now: 70 and without the biggest orders: 70
customers in both groups: 68
customers who leave the At risk group: ['C0102', 'C0198']
customers who join the At risk group: ['C0321', 'C0408']
firm At risk customers still At risk without their biggest order: 21 of 21


In [27]:
# count months with orders, then rebuild the segments as they stood on 1 January 2024 and compare them with today's segments
order_dates = pd.to_datetime(sales_h["order_date"])
per_month = order_dates.dt.to_period("M").value_counts().sort_index()
print("months with Delivered orders:", len(per_month), "first month:", per_month.index.min(), "last month:", per_month.index.max())
print("fewest orders in a month:", per_month.min(), "most orders in a month:", per_month.max())
print("orders added over all months:", per_month.sum())
print()

cut = pd.Timestamp("2024-01-01")
past = sales_h[order_dates < cut].copy()
past["order_date"] = pd.to_datetime(past["order_date"])
then = past.groupby("customer_id").agg(last_order=("order_date", "max"), frequency=("so_id", "count"), monetary=("total_order_value", "sum"))
then["recency_days"] = (cut - then["last_order"]).dt.days
print("customers with orders before 1 January 2024:", len(then), "of 500")

r_then = 6 - group_of(then["recency_days"].values)
f_then = group_of(then["frequency"].values)
m_then = group_of(then["monetary"].values)
then["segment_then"] = [segment_with(r_then[i], f_then[i], m_then[i]) for i in range(len(then))]
then["segment_now"] = rfm.set_index("customer_id").loc[then.index, "segment"].values

print("customers with the same segment a year later:", (then["segment_then"] == then["segment_now"]).sum(), "of", len(then))
print("customers At risk on 1 January 2024:", (then["segment_then"] == at_risk).sum())
print("of today's At risk customers, already At risk on 1 January 2024:", ((then["segment_then"] == at_risk) & (then["segment_now"] == at_risk)).sum(), "of", (then["segment_now"] == at_risk).sum())
print()
print(pd.crosstab(then["segment_then"], then["segment_now"]).to_string())

months with Delivered orders: 72 first month: 2019-01 last month: 2024-12
fewest orders in a month: 209 most orders in a month: 312
orders added over all months: 18033

customers with orders before 1 January 2024: 500 of 500
customers with the same segment a year later: 155 of 500
customers At risk on 1 January 2024: 75
of today's At risk customers, already At risk on 1 January 2024: 22 of 70

segment_now                   At risk (valuable but quiet)  Average  Champions  Loyal  Quiet and low value  Recent but small
segment_then                                                                                                                
At risk (valuable but quiet)                            22        1         22     16                   10                 4
Average                                                  1       12          0      1                   23                23
Champions                                               23        2         27     18                  

In [28]:
# compare each score a year ago with today to see which part of the segment moves
now_scores = rfm.set_index("customer_id").loc[then.index]
for name, before, after in [("recency", r_then, now_scores["R_score"].values), ("frequency", f_then, now_scores["F_score"].values), ("money", m_then, now_scores["M_score"].values)]:
    print(name, "score: same score a year later for", (before == after).sum(), "of", len(then), "customers, within one step for", (np.abs(before - after) <= 1).sum())
    print("  rank correlation between a year ago and today:", round(pd.Series(before).corr(pd.Series(after), method="spearman"), 2))

recency score: same score a year later for 94 of 500 customers, within one step for 249
  rank correlation between a year ago and today: -0.01
frequency score: same score a year later for 315 of 500 customers, within one step for 490
  rank correlation between a year ago and today: 0.89
money score: same score a year later for 288 of 500 customers, within one step for 480
  rank correlation between a year ago and today: 0.86


In [29]:
# compare each customer's orders and money in 2023 with 2024 (different orders in each year) and compare the spread of order counts with chance
year_of_order = pd.to_datetime(sales_h["order_date"]).dt.year
two_years = sales_h[year_of_order.isin([2023, 2024])].assign(year=year_of_order)
year_table = two_years.pivot_table(index="customer_id", columns="year", values="total_order_value", aggfunc=["count", "sum"]).fillna(0)
orders_2023 = year_table[("count", 2023)]
orders_2024 = year_table[("count", 2024)]
money_2023 = year_table[("sum", 2023)]
money_2024 = year_table[("sum", 2024)]
print("customers with orders in 2023 or 2024:", len(year_table), "of 500")
print("orders per customer in 2023: average", round(orders_2023.mean(), 2), "spread", round(orders_2023.std(), 2))
print("orders per customer in 2024: average", round(orders_2024.mean(), 2), "spread", round(orders_2024.std(), 2))
print("rank correlation of orders, 2023 against 2024:", round(orders_2023.corr(orders_2024, method="spearman"), 2))
print("rank correlation of money, 2023 against 2024:", round(money_2023.corr(money_2024, method="spearman"), 2))
all_years = rfm["frequency"]
print("orders over all six years: average", round(all_years.mean(), 2), "spread", round(all_years.std(), 2))
print("spread expected from chance alone (square root of the average):", round(all_years.mean() ** 0.5, 2))

customers with orders in 2023 or 2024: 500 of 500
orders per customer in 2023: average 6.04 spread 2.5
orders per customer in 2024: average 5.91 spread 2.42
rank correlation of orders, 2023 against 2024: -0.04
rank correlation of money, 2023 against 2024: 0.04
orders over all six years: average 36.07 spread 6.04
spread expected from chance alone (square root of the average): 6.01


In [30]:
# simulate 500 customers who order completely at random at the same average rate and compare with the real result
rng = np.random.default_rng(42)
days_in_history = (pd.Timestamp("2025-01-01") - pd.Timestamp("2019-01-01")).days
average_orders = rfm["frequency"].mean()
real_order_values = sales_h["total_order_value"].values

quiet_counts = []
at_risk_counts = []
spreads = []
for run in range(200):
    sim_recency, sim_orders, sim_money, sim_ratio = [], [], [], []
    for customer in range(500):
        n = max(rng.poisson(average_orders), 2)
        days = np.sort(rng.integers(0, days_in_history, size=n))
        gap = (days[-1] - days[0]) / (n - 1)
        sim_recency.append(days_in_history - days[-1])
        sim_orders.append(n)
        sim_money.append(rng.choice(real_order_values, size=n).sum())
        sim_ratio.append((days_in_history - days[-1]) / gap)
    quiet_counts.append((np.array(sim_ratio) > 2).sum())
    spreads.append(np.std(sim_orders, ddof=1))
    r_sim = 6 - group_of(np.array(sim_recency))
    f_sim = group_of(np.array(sim_orders))
    m_sim = group_of(np.array(sim_money))
    at_risk_counts.append(sum(segment_with(r_sim[i], f_sim[i], m_sim[i]) == at_risk for i in range(500)))

print("customers quiet beyond twice their own normal gap: real 67, random customers average", round(np.mean(quiet_counts), 1), "lowest", min(quiet_counts), "highest", max(quiet_counts))
print("customers in the At risk group: real 70, random customers average", round(np.mean(at_risk_counts), 1), "lowest", min(at_risk_counts), "highest", max(at_risk_counts))
print("spread of order counts: real 6.04, random customers average", round(np.mean(spreads), 2))

customers quiet beyond twice their own normal gap: real 67, random customers average 72.0 lowest 51 highest 93
customers in the At risk group: real 70, random customers average 73.7 lowest 54 highest 88
spread of order counts: real 6.04, random customers average 5.98


In [31]:
# move the snapshot date earlier and later and see how many customers change segment and how the quiet count changes
def segments_at(snapshot):
    all_dates = pd.to_datetime(sales_h["order_date"])
    part = sales_h[all_dates < snapshot].assign(order_date=all_dates[all_dates < snapshot])
    t = part.groupby("customer_id").agg(first_order=("order_date", "min"), last_order=("order_date", "max"), frequency=("so_id", "count"), monetary=("total_order_value", "sum"))
    t["recency_days"] = (snapshot - t["last_order"]).dt.days
    r = 6 - group_of(t["recency_days"].values)
    f = group_of(t["frequency"].values)
    m = group_of(t["monetary"].values)
    t["segment"] = [segment_with(r[i], f[i], m[i]) for i in range(len(t))]
    t["gap"] = (t["last_order"] - t["first_order"]).dt.days / (t["frequency"] - 1)
    t["ratio"] = t["recency_days"] / t["gap"]
    return t

table_now = rfm.set_index("customer_id")
for day in ["2024-07-01", "2024-12-01", "2025-01-01", "2025-01-31"]:
    t = segments_at(pd.Timestamp(day))
    changed = (t["segment"] != table_now.loc[t.index, "segment"]).sum()
    print("snapshot", day, ": customers", len(t), "| change segment vs our table", changed, "| At risk group", (t["segment"] == at_risk).sum(), "| quiet beyond twice their normal gap", (t["ratio"] > 2).sum())

snapshot 2024-07-01 : customers 500 | change segment vs our table 340 | At risk group 69 | quiet beyond twice their normal gap 69
snapshot 2024-12-01 : customers 500 | change segment vs our table 270 | At risk group 66 | quiet beyond twice their normal gap 69
snapshot 2025-01-01 : customers 500 | change segment vs our table 0 | At risk group 70 | quiet beyond twice their normal gap 67
snapshot 2025-01-31 : customers 500 | change segment vs our table 0 | At risk group 70 | quiet beyond twice their normal gap 123


In [32]:
# check three customers by hand from the raw files: C0001, C0002 and C0003
import csv
from datetime import date

wanted = ["C0001", "C0002", "C0003"]
orders_raw = {c: [] for c in wanted}
with open(BASE_PATH + "sales_orders_header.csv", newline="") as f:
    for line in csv.DictReader(f):
        if line["customer_id"] in orders_raw and line["order_status"] == "Delivered":
            orders_raw[line["customer_id"]].append((line["so_id"], date.fromisoformat(line["order_date"][:10]), float(line["total_order_value"])))

order_owner = {so: c for c, items in orders_raw.items() for (so, d, v) in items}
lines_money = {c: 0.0 for c in wanted}
with open(BASE_PATH + "sales_orders_lines.csv", newline="") as f:
    for line in csv.DictReader(f):
        if line["so_id"] in order_owner:
            lines_money[order_owner[line["so_id"]]] += float(line["line_total"])

snapshot_day = date(2025, 1, 1)
table_by_customer = rfm.set_index("customer_id")
for c in wanted:
    dates = sorted(d for (so, d, v) in orders_raw[c])
    n = len(dates)
    gap = (dates[-1] - dates[0]).days / (n - 1)
    recency = (snapshot_day - dates[-1]).days
    print(c, "from the raw files: orders", n, "| first order", dates[0], "| last order", dates[-1], "| recency days", recency, "| money from headers", sum(v for (so, d, v) in orders_raw[c]), "| money from lines", lines_money[c], "| average gap", round(gap, 2), "| silence ratio", round(recency / gap, 2))
    t = table_by_customer.loc[c]
    print(c, "from our table:      orders", t["frequency"], "| first order", t["first_order"], "| last order", t["last_order"], "| recency days", t["recency_days"], "| money", t["monetary"], "| average gap", round(t["avg_gap_days"], 2), "| silence ratio", round(t["silence_vs_gap"], 2), "| scores R, F, M:", t["R_score"], t["F_score"], t["M_score"], "| segment", t["segment"])
    print()

C0001 from the raw files: orders 34 | first order 2019-01-26 | last order 2024-11-15 | recency days 47 | money from headers 43201680.0 | money from lines 43201680.0 | average gap 64.24 | silence ratio 0.73
C0001 from our table:      orders 34 | first order 2019-01-26 00:00:00 | last order 2024-11-15 00:00:00 | recency days 47 | money 43201680 | average gap 64.24 | silence ratio 0.73 | scores R, F, M: 3 2 3 | segment Average

C0002 from the raw files: orders 51 | first order 2019-01-24 | last order 2024-09-07 | recency days 116 | money from headers 56261100.0 | money from lines 56261100.0 | average gap 41.06 | silence ratio 2.83
C0002 from our table:      orders 51 | first order 2019-01-24 00:00:00 | last order 2024-09-07 00:00:00 | recency days 116 | money 56261100 | average gap 41.06 | silence ratio 2.83 | scores R, F, M: 1 5 5 | segment At risk (valuable but quiet)

C0003 from the raw files: orders 38 | first order 2019-04-05 | last order 2024-10-30 | recency days 63 | money from hea